**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Reinforcement Learning

Learning from *consequences* instead of labels — the paradigm the [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) name-dropped as RLHF and this course delivers. Five sessions: bandits, MDPs & Bellman, temporal-difference learning, policy gradients, and the road to RLHF — every algorithm verified against an exactly-solvable environment.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb) (expectations, LLN).
- [Training Dynamics](./Training_Dynamics.ipynb) for Session 4.
- Kinship worth knowing: TD learning's *new = old + α·(surprise)* is the [adaptive-filter heartbeat](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) yet again.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 5 — *Bandits: Exploration vs Exploitation* (~35 min)
**Goal:** the RL problem with no states: regret, ε-greedy, and UCB's optimism.
**Feeds into:** Session 2 (MDPs).

---

## 2. The Ten-Armed Testbed

💡 **Intuition.** Ten slot machines, unknown payouts, 1000 pulls: every pull spent *learning* is a pull not spent *earning*. That tension — exploration vs exploitation — is RL's signature dilemma, isolated from everything else. **ε-greedy** explores blindly and forever; **UCB** explores *strategically*: pull the arm whose plausible upside $\hat\mu_a + c\sqrt{\ln t / n_a}$ is highest — 'optimism in the face of uncertainty', with the bonus shrinking exactly like a [confidence interval](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [2]:
def bandit_run(policy, T=1000, K=10, runs=800):
    regret = np.zeros(T)
    for r in range(runs):
        mu = rng.standard_normal(K)
        best = mu.max()
        Q, N = np.zeros(K), np.zeros(K)
        for t in range(T):
            a = policy(Q, N, t)
            reward = mu[a] + rng.standard_normal()
            N[a] += 1; Q[a] += (reward - Q[a]) / N[a]          # incremental mean (the heartbeat)
            regret[t] += best - mu[a]
    return np.cumsum(regret / runs)

eps_greedy = lambda eps: (lambda Q, N, t: rng.integers(len(Q)) if rng.random() < eps else int(np.argmax(Q)))
def ucb(Q, N, t):
    if (N == 0).any(): return int(np.argmax(N == 0))
    return int(np.argmax(Q + 2.0 * np.sqrt(np.log(t + 1) / N)))

plt.figure(figsize=(8, 3))
for name, pol in [("greedy (ε=0)", eps_greedy(0)), ("ε=0.1", eps_greedy(0.1)), ("UCB", ucb)]:
    r = bandit_run(pol)
    plt.plot(r, label=f"{name}: total regret {r[-1]:.0f}")
plt.legend(); plt.xlabel("pull"); plt.ylabel("cumulative regret")
plt.title("greedy plateaus on a wrong arm; ε keeps paying tax; UCB's tax shrinks")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2966289/3262936799.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 5 — *MDPs & the Bellman Equation* (~40 min)
**Goal:** add states and time; solve a gridworld EXACTLY by dynamic programming.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (TD learning).

---

## 3. Markov Decision Processes

💡 **Intuition.** Now actions have *consequences that persist*: an MDP is states, actions, transition probabilities, rewards, and a discount $\gamma$. The value $V^\pi(s)$ is expected discounted return — and Bellman's equation says value is **recursively self-consistent**: today's value = today's reward + γ·tomorrow's value. The optimal version ($V^* = \max_a [r + \gamma E V^*]$) is a fixed-point equation, and **value iteration** just applies it until it stops moving — a contraction ([Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb): Cauchy convergence with rate γ!). This gives us an *exact oracle* to test every learning algorithm against.

In [3]:
# 4x4 gridworld: start anywhere, goal at (3,3) reward +1, pit at (1,2) reward −1, step −0.02
SIZE, GOAL, PIT = 4, 15, 6
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
gamma, slip = 0.95, 0.1                                  # 10% chance the move slips sideways

def step_model(s, a):
    """returns list of (prob, s', r, done)"""
    if s in (GOAL, PIT): return [(1.0, s, 0.0, True)]
    out = []
    for prob, ai in [(1-slip, a), (slip/2, (a+2)%4), (slip/2, (a+3)%4 if a%2 else (a+1)%4)]:
        r0, c0 = divmod(s, SIZE)
        dr, dc = ACTIONS[ai]
        r1, c1 = min(max(r0+dr,0),SIZE-1), min(max(c0+dc,0),SIZE-1)
        s1 = r1*SIZE + c1
        rew = 1.0 if s1 == GOAL else (-1.0 if s1 == PIT else -0.02)
        out.append((prob, s1, rew, s1 in (GOAL, PIT)))
    return out

# value iteration = the exact solution (our ORACLE for everything later)
V = np.zeros(16)
for it in range(500):
    V_new = np.array([max(sum(p*(r + gamma*V[s1]*(not d)) for p, s1, r, d in step_model(s, a))
                          for a in range(4)) for s in range(16)])
    if np.abs(V_new - V).max() < 1e-12: break
    V = V_new
Q_star = np.array([[sum(p*(r + gamma*V[s1]*(not d)) for p, s1, r, d in step_model(s, a))
                    for a in range(4)] for s in range(16)])
pi_star = Q_star.argmax(1)
print(f"value iteration converged in {it} sweeps (contraction at rate γ={gamma})")
arrows = np.array(["↑","↓","←","→"])[pi_star].reshape(4,4)
arrows[3,3] = "G"; arrows[1,2] = "P"
print("optimal policy:\n", arrows)

value iteration converged in 32 sweeps (contraction at rate γ=0.95)
optimal policy:
 [['↓' '↓' '→' '↓']
 ['↓' '↓' 'P' '↓']
 ['→' '→' '→' '↓']
 ['→' '→' '→' 'G']]


---
### 🕐 Session 3 of 5 — *Temporal-Difference Learning* (~40 min)
**Goal:** learn the same values WITHOUT the model: TD(0) and Q-learning, checked against the oracle.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (policy gradients).

---

## 4. Learning from the Surprise

💡 **Intuition.** Value iteration needed the transition model. An *agent* only gets experience: $(s, a, r, s')$. TD's move: use the Bellman equation as an **error signal** — the *TD error* $\delta = r + \gamma \max_a Q(s', a) - Q(s, a)$ is how surprised you are, and $Q \mathrel{+}= \alpha \delta$ is the [LMS update](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup as the 'desired signal'. Q-learning does this off-policy (learns the greedy value while exploring) — and, on a finite MDP with decaying exploration, provably converges to $Q^*$. We *check* that, since we own the oracle.

In [4]:
def env_step(s, a):
    outs = step_model(s, a)
    probs = [o[0] for o in outs]
    _, s1, r, d = outs[rng.choice(len(outs), p=probs)]
    return s1, r, d

Q = np.zeros((16, 4))
N_sa = np.zeros((16, 4))                                   # visit counts → decaying step size
eps = 1.0
errs = []
for ep in range(12000):
    s = rng.choice([s for s in range(16) if s not in (GOAL, PIT)])
    eps = max(0.05, eps * 0.9995)
    for _ in range(100):
        a = rng.integers(4) if rng.random() < eps else int(Q[s].argmax())
        s1, r, done = env_step(s, a)
        target = r + (0 if done else gamma * Q[s1].max())
        N_sa[s, a] += 1
        alpha = 1.0 / N_sa[s, a]**0.6                      # Robbins–Monro: Σα=∞, Σα²<∞ → convergence
        Q[s, a] += alpha * (target - Q[s, a])              # new = old + α·(surprise)
        s = s1
        if done: break
    if ep % 200 == 0: errs.append(np.abs(Q - Q_star).max())

plt.figure(figsize=(7.5, 2.6))
plt.semilogy(np.arange(len(errs))*200, errs)
plt.xlabel("episode"); plt.ylabel("‖Q − Q*‖∞")
plt.title("Q-learning converges to the DP oracle's Q* — from experience alone")
plt.tight_layout(); plt.show()
# tie-aware policy check: the learned greedy action must be (near-)optimal under Q*
nonterm = [s for s in range(16) if s not in (GOAL, PIT)]
optimal_choice = np.array([Q_star[s, Q[s].argmax()] >= Q_star[s].max() - 1e-3 for s in nonterm])
print(f"final ‖Q − Q*‖∞ = {np.abs(Q - Q_star).max():.3f}; learned greedy action optimal in {optimal_choice.mean():.0%} of states")

final ‖Q − Q*‖∞ = 0.232; learned greedy action optimal in 100% of states


/tmp/ipykernel_2966289/1394493565.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 5 — *Policy Gradients* (~40 min)
**Goal:** skip values, optimize the policy directly: REINFORCE with a baseline, from scratch.
**Builds on:** Session 3; [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 5 (the road to RLHF).

---

## 5. Differentiating Through Luck

💡 **Intuition.** Values are a detour; why not adjust the policy's parameters to make good episodes more likely? The log-derivative trick makes the un-differentiable differentiable: $\nabla E[R] = E[R \, \nabla \log \pi(a|s)]$ — *reinforce the log-probability of what you did, in proportion to how well it went*. The estimator is unbiased but wildly noisy ([SGD's](../Intro_Math/Optimization/Optimization.ipynb) noise-floor problem, squared); subtracting a **baseline** (the mean return) cancels variance without adding bias — the single most important practical trick in policy-land.

In [5]:
# REINFORCE on the same gridworld (tabular softmax policy)
def run_episode(theta, max_steps=60):
    s = 0; traj = []
    for _ in range(max_steps):
        logits = theta[s]
        p = np.exp(logits - logits.max()); p /= p.sum()
        a = rng.choice(4, p=p)
        s1, r, done = env_step(s, a)
        traj.append((s, a, r))
        s = s1
        if done: break
    return traj

def reinforce(use_baseline, iters=1500, lr=0.15):
    theta = np.zeros((16, 4)); returns_hist = []
    for it in range(iters):
        traj = run_episode(theta)
        G = 0.0; Gs = []
        for (_, _, r) in reversed(traj):
            G = r + gamma * G; Gs.append(G)
        Gs = Gs[::-1]
        b = np.mean(Gs) if use_baseline else 0.0
        for (s, a, _), G_t in zip(traj, Gs):
            p = np.exp(theta[s] - theta[s].max()); p /= p.sum()
            grad = -p; grad[a] += 1                       # ∇ log softmax
            theta[s] += lr * (G_t - b) * grad
        returns_hist.append(Gs[0])
    return np.array(returns_hist)

plt.figure(figsize=(8, 2.8))
for name, ub in [("REINFORCE", False), ("REINFORCE + baseline", True)]:
    h = reinforce(ub)
    sm = np.convolve(h, np.ones(50)/50, "valid")
    plt.plot(sm, label=f"{name} (final ≈ {sm[-1]:.2f})")
plt.axhline(V[0], color="k", linestyle=":", linewidth=0.9, label=f"oracle V*(start) = {V[0]:.2f}")
plt.legend(fontsize=8); plt.xlabel("episode"); plt.ylabel("return from start")
plt.title("policy gradient reaches the DP oracle's value; the baseline gets there calmer")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2966289/2260655531.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 5 of 5 — *The Road to RLHF* (~30 min)
**Goal:** connect this course to modern practice: reward models, KL anchors, and PPO's role.
**Builds on:** Session 4.

---

## 6. From Gridworld to Chatbots

The [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) said 'preference tuning shapes judgment'; you now have the vocabulary for how:

1. **The policy** is the language model; a *state* is the prompt + text so far, an *action* is the next token, an *episode* is a completion.
2. **The reward** comes from a *reward model* trained on human preference pairs — [cross-entropy](../Intro_Math/Information_Theory/Information_Theory.ipynb) on 'which answer did the human prefer'.
3. **The optimizer** is a policy gradient with variance-reduction armor: PPO ≈ REINFORCE + a learned baseline (critic) + a *trust region* (clipped updates — don't move the policy further than the reward model's validity extends).
4. **The KL anchor**: reward is penalized by KL divergence from the pretrained model — 'improve preferences *without leaving the language manifold*'. DPO folds reward model + RL into one supervised loss on preference pairs, which is why it took over.

💡 **Intuition.** Everything hard about RLHF is Session 4's variance problem wearing a $10^{11}$-parameter costume, plus one new failure mode this course equips you to name: **reward hacking** — the policy exploiting the reward model where it's [off-distribution](./Uncertainty_in_ML.ipynb).

## 7. Conclusion

Bandits isolate exploration; Bellman makes value self-consistent; TD learns it from surprise (LMS's heartbeat again); policy gradients differentiate through luck with a baseline as armor; RLHF is all four at industrial scale. Every algorithm here was checked against an exact oracle — a habit worth keeping when the environments stop being 4×4.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — the policy being tuned.
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — the SGD theory under the noise.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-state tracking: what 'state' means when you can't see it.